<a href="https://colab.research.google.com/github/Muktesh9-11/ecommerce-data-engineering/blob/main/04_silver_transformations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install only if PySpark is not already available
!pip install pyspark -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

spark = (
    SparkSession.builder
    .appName("Olist-Silver-Transformations")
    .master("local[*]")
    .getOrCreate()
)

PROJECT = Path("/content/drive/MyDrive/ecommerce-data-engineering")

BRONZE = PROJECT / "data" / "bronze"
SILVER = PROJECT / "data" / "silver"
REPORTS = PROJECT / "reports"

SILVER.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print("Spark version:", spark.version)
print("Bronze path:", BRONZE)
print("Silver path:", SILVER)

Mounted at /content/drive
Spark version: 4.0.5
Bronze path: /content/drive/MyDrive/ecommerce-data-engineering/data/bronze
Silver path: /content/drive/MyDrive/ecommerce-data-engineering/data/silver


In [ ]:
TYPE_MAPPING = {
    "olist_geolocation_dataset": {
        "geolocation_lat": "double",
        "geolocation_lng": "double"
    },

    "olist_order_items_dataset": {
        "order_item_id": "int",
        "shipping_limit_date": "timestamp",
        "price": "decimal(12,2)",
        "freight_value": "decimal(12,2)"
    },

    "olist_order_payments_dataset": {
        "payment_sequential": "int",
        "payment_installments": "int",
        "payment_value": "decimal(12,2)"
    },

    "olist_order_reviews_dataset": {
        "review_score": "int",
        "review_creation_date": "timestamp",
        "review_answer_timestamp": "timestamp"
    },

    "olist_orders_dataset": {
        "order_purchase_timestamp": "timestamp",
        "order_approved_at": "timestamp",
        "order_delivered_carrier_date": "timestamp",
        "order_delivered_customer_date": "timestamp",
        "order_estimated_delivery_date": "timestamp"
    },

    "olist_products_dataset": {
        "product_name_lenght": "int",
        "product_description_lenght": "int",
        "product_photos_qty": "int",
        "product_weight_g": "int",
        "product_length_cm": "int",
        "product_height_cm": "int",
        "product_width_cm": "int"
    }
}

In [ ]:
def transform_silver(df, table_name):

    # 1. Trim whitespace and convert empty strings to NULL
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            c = field.name

            df = df.withColumn(
                c,
                F.when(
                    F.trim(F.col(c)) == "",
                    F.lit(None)
                ).otherwise(F.trim(F.col(c)))
            )

    # 2. Apply explicit data types
    casts = TYPE_MAPPING.get(table_name, {})

    for column_name, data_type in casts.items():
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                F.expr(
                    f"try_cast(`{column_name}` AS {data_type})"
                )
            )

    # 3. Remove exact duplicate geolocation records only
    if table_name == "olist_geolocation_dataset":
        before = df.count()
        df = df.dropDuplicates()
        after = df.count()

        print(
            f"Geolocation exact duplicates removed: "
            f"{before - after:,}"
        )

    # 4. Add Silver processing metadata
    df = df.withColumn(
        "_silver_processed_at_utc",
        F.current_timestamp()
    )

    return df

In [ ]:
bronze_tables = sorted(
    path.name for path in BRONZE.iterdir()
    if path.is_dir()
)

silver_audit = []
silver_dfs = {}

for table_name in bronze_tables:

    print(f"\nProcessing: {table_name}")

    bronze_path = BRONZE / table_name
    silver_path = SILVER / table_name

    # Read Bronze
    bronze_df = spark.read.parquet(str(bronze_path))
    bronze_count = bronze_df.count()

    # Transform
    silver_df = transform_silver(
        bronze_df,
        table_name
    )

    # Write Silver
    silver_df.coalesce(2).write.mode("overwrite").parquet(
        str(silver_path)
    )

    # Read back and validate
    written_df = spark.read.parquet(str(silver_path))
    silver_count = written_df.count()

    silver_dfs[table_name] = written_df

    silver_audit.append({
        "table_name": table_name,
        "bronze_records": bronze_count,
        "silver_records": silver_count,
        "records_removed": bronze_count - silver_count,
        "bronze_columns": len(bronze_df.columns),
        "silver_columns": len(written_df.columns),
        "output_path": str(silver_path),
        "status": "SUCCESS"
    })

    print(
        f"Bronze: {bronze_count:,} | "
        f"Silver: {silver_count:,} | "
        f"Removed: {bronze_count - silver_count:,}"
    )

print("\nAll Silver transformations completed.")


Processing: olist_customers_dataset
Bronze: 99,441 | Silver: 99,441 | Removed: 0

Processing: olist_geolocation_dataset
Geolocation exact duplicates removed: 261,831
Bronze: 1,000,163 | Silver: 738,332 | Removed: 261,831

Processing: olist_order_items_dataset
Bronze: 112,650 | Silver: 112,650 | Removed: 0

Processing: olist_order_payments_dataset
Bronze: 103,886 | Silver: 103,886 | Removed: 0

Processing: olist_order_reviews_dataset
Bronze: 99,249 | Silver: 99,249 | Removed: 0

Processing: olist_orders_dataset
Bronze: 99,441 | Silver: 99,441 | Removed: 0

Processing: olist_products_dataset
Bronze: 32,951 | Silver: 32,951 | Removed: 0

Processing: olist_sellers_dataset
Bronze: 3,095 | Silver: 3,095 | Removed: 0

Processing: product_category_name_translation
Bronze: 71 | Silver: 71 | Removed: 0

All Silver transformations completed.


In [ ]:
import pandas as pd

silver_audit_df = pd.DataFrame(silver_audit)

display(silver_audit_df)

silver_audit_path = REPORTS / "silver_ingestion_audit.csv"

silver_audit_df.to_csv(
    silver_audit_path,
    index=False
)

print("Total Bronze records:",
      f"{silver_audit_df['bronze_records'].sum():,}")

print("Total Silver records:",
      f"{silver_audit_df['silver_records'].sum():,}")

print("Total records removed:",
      f"{silver_audit_df['records_removed'].sum():,}")

print("All tables successful:",
      (silver_audit_df["status"] == "SUCCESS").all())

print("Audit saved to:", silver_audit_path)

,table_name,bronze_records,silver_records,records_removed,bronze_columns,silver_columns,output_path,status
0,olist_customers_dataset,99441,99441,0,8,9,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
1,olist_geolocation_dataset,1000163,738332,261831,8,9,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
2,olist_order_items_dataset,112650,112650,0,10,11,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
3,olist_order_payments_dataset,103886,103886,0,8,9,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
4,olist_order_reviews_dataset,99249,99249,0,10,11,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
5,olist_orders_dataset,99441,99441,0,11,12,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
6,olist_products_dataset,32951,32951,0,12,13,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
7,olist_sellers_dataset,3095,3095,0,7,8,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS
8,product_category_name_translation,71,71,0,5,6,/content/drive/MyDrive/ecommerce-data-engineer...,SUCCESS


Total Bronze records: 1,550,947
Total Silver records: 1,289,116
Total records removed: 261,831
All tables successful: True
Audit saved to: /content/drive/MyDrive/ecommerce-data-engineering/reports/silver_ingestion_audit.csv


In [ ]:
from functools import reduce
from operator import or_

KEY_COLUMNS = {
    "olist_customers_dataset": ["customer_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": ["product_category_name"]
}

key_quality = []

for table_name, keys in KEY_COLUMNS.items():

    df = silver_dfs[table_name]

    # Count rows with at least one null key component
    null_condition = reduce(
        or_,
        [F.col(k).isNull() for k in keys]
    )

    null_key_rows = df.filter(null_condition).count()

    # Find duplicate key combinations
    duplicate_groups = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_group_count = duplicate_groups.count()

    duplicate_extra_rows = (
        duplicate_groups
        .agg(
            F.sum(F.col("count") - 1).alias("extra_rows")
        )
        .first()["extra_rows"]
        or 0
    )

    key_quality.append({
        "table_name": table_name,
        "key_columns": ", ".join(keys),
        "null_key_rows": null_key_rows,
        "duplicate_key_groups": duplicate_group_count,
        "duplicate_extra_rows": int(duplicate_extra_rows)
    })

key_quality_df = pd.DataFrame(key_quality)

display(key_quality_df)

key_quality_df.to_csv(
    REPORTS / "silver_key_quality_report.csv",
    index=False
)

,table_name,key_columns,null_key_rows,duplicate_key_groups,duplicate_extra_rows
0,olist_customers_dataset,customer_id,0,0,0
1,olist_orders_dataset,order_id,0,0,0
2,olist_order_items_dataset,"order_id, order_item_id",0,0,0
3,olist_order_payments_dataset,"order_id, payment_sequential",0,0,0
4,olist_order_reviews_dataset,review_id,0,789,814
5,olist_products_dataset,product_id,0,0,0
6,olist_sellers_dataset,seller_id,0,0,0
7,product_category_name_translation,product_category_name,0,0,0


In [ ]:
RELATIONSHIPS = [
    ("olist_orders_dataset", "customer_id",
     "olist_customers_dataset", "customer_id"),

    ("olist_order_items_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("olist_order_items_dataset", "product_id",
     "olist_products_dataset", "product_id"),

    ("olist_order_items_dataset", "seller_id",
     "olist_sellers_dataset", "seller_id"),

    ("olist_order_payments_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("olist_order_reviews_dataset", "order_id",
     "olist_orders_dataset", "order_id")
]

relationship_results = []

for child_table, child_key, parent_table, parent_key in RELATIONSHIPS:

    child = silver_dfs[child_table].alias("c")

    parent_keys = (
        silver_dfs[parent_table]
        .select(F.col(parent_key).alias("parent_key"))
        .filter(F.col("parent_key").isNotNull())
        .distinct()
        .alias("p")
    )

    unmatched = (
        child
        .filter(F.col(f"c.{child_key}").isNotNull())
        .join(
            parent_keys,
            F.col(f"c.{child_key}") == F.col("p.parent_key"),
            "left_anti"
        )
    )

    unmatched_count = unmatched.count()

    relationship_results.append({
        "child_table": child_table,
        "child_key": child_key,
        "parent_table": parent_table,
        "parent_key": parent_key,
        "unmatched_records": unmatched_count,
        "status": "PASS" if unmatched_count == 0 else "REVIEW"
    })

relationship_df = pd.DataFrame(relationship_results)

display(relationship_df)

relationship_df.to_csv(
    REPORTS / "silver_relationship_report.csv",
    index=False
)

,child_table,child_key,parent_table,parent_key,unmatched_records,status
0,olist_orders_dataset,customer_id,olist_customers_dataset,customer_id,0,PASS
1,olist_order_items_dataset,order_id,olist_orders_dataset,order_id,0,PASS
2,olist_order_items_dataset,product_id,olist_products_dataset,product_id,0,PASS
3,olist_order_items_dataset,seller_id,olist_sellers_dataset,seller_id,0,PASS
4,olist_order_payments_dataset,order_id,olist_orders_dataset,order_id,0,PASS
5,olist_order_reviews_dataset,order_id,olist_orders_dataset,order_id,17,REVIEW


In [ ]:
silver_orders = silver_dfs["olist_orders_dataset"]

silver_orders.printSchema()
silver_orders.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at_utc: timestamp (nullable = true)
 |-- _bronze_table: string (nullable = true)
 |-- _silver_processed_at_utc: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+------------------------+-------------------------+--------------------+--------------------------+
|order_id                        |customer_id       

In [ ]:
silver_reviews = silver_dfs["olist_order_reviews_dataset"]

silver_reviews.printSchema()
silver_reviews.show(5, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at_utc: timestamp (nullable = true)
 |-- _bronze_table: string (nullable = true)
 |-- _silver_processed_at_utc: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+-------------------------------+--------------------------+---------------------------+--------------------------+
|review_id                       |order_id                        |review_score|review_c

In [ ]:
print("===== KEY QUALITY REPORT =====")
print(key_quality_df.to_string(index=False))

print("\n===== RELATIONSHIP REPORT =====")
print(relationship_df.to_string(index=False))

===== KEY QUALITY REPORT =====
                       table_name                  key_columns  null_key_rows  duplicate_key_groups  duplicate_extra_rows
          olist_customers_dataset                  customer_id              0                     0                     0
             olist_orders_dataset                     order_id              0                     0                     0
        olist_order_items_dataset      order_id, order_item_id              0                     0                     0
     olist_order_payments_dataset order_id, payment_sequential              0                     0                     0
      olist_order_reviews_dataset                    review_id              0                   789                   814
           olist_products_dataset                   product_id              0                     0                     0
            olist_sellers_dataset                    seller_id              0                     0                

In [ ]:
reviews = silver_dfs["olist_order_reviews_dataset"]

duplicate_review_ids = (
    reviews.groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate review ID groups:", duplicate_review_ids.count())

# Display examples with their complete review details
duplicate_review_records = (
    reviews.join(
        duplicate_review_ids.select("review_id"),
        on="review_id",
        how="inner"
    )
    .orderBy("review_id")
)

duplicate_review_records.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(30, truncate=False)

Duplicate review ID groups: 789
+--------------------------------+--------------------------------+------------+--------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                                                  |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
review_business_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

exact_duplicate_excess = (
    reviews.select(*review_business_columns)
    .groupBy(*review_business_columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.sum(F.col("count") - 1).alias("duplicate_rows")
    )
    .first()["duplicate_rows"]
)

print("Exact duplicate review records:", exact_duplicate_excess or 0)

Exact duplicate review records: 0


In [ ]:
orders = silver_dfs["olist_orders_dataset"]

valid_order_ids = (
    orders.select("order_id")
    .distinct()
)

unmatched_reviews = (
    reviews.alias("r")
    .join(
        valid_order_ids.alias("o"),
        F.col("r.order_id") == F.col("o.order_id"),
        "left_anti"
    )
)

print("Unmatched review records:", unmatched_reviews.count())

unmatched_reviews.select(
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(30, truncate=False)

Unmatched review records: 26
+-------------------------------------------------------------------------------------------+------------------------------------+------------+--------------------+-----------------------+
|review_id                                                                                  |order_id                            |review_score|review_creation_date|review_answer_timestamp|
+-------------------------------------------------------------------------------------------+------------------------------------+------------+--------------------+-----------------------+
|Ficou a dúvida se o cartucho é original."                                                  |2018-06-29 00:00:00                 |NULL        |NULL                |NULL                   |
|Assim                                                                                      |solicito esclarecimentos."          |NULL        |NULL                |NULL                   |
|Esta loja eu recomendo   

In [ ]:
RAW_PATH = "/content/drive/MyDrive/ecommerce-data-engineering/data/raw"

REPORTS_PATH = "/content/drive/MyDrive/ecommerce-data-engineering/reports"

In [ ]:
reviews_path = f"{RAW_PATH}/olist_order_reviews_dataset.csv"

reviews_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(reviews_path)
)

print("Review count:", reviews_test.count())

print(
    "Invalid review IDs:",
    reviews_test.filter(
        ~F.col("review_id").rlike("^[0-9a-fA-F]{32}$")
    ).count()
)

print(
    "Invalid order IDs:",
    reviews_test.filter(
        ~F.col("order_id").rlike("^[0-9a-fA-F]{32}$")
    ).count()
)

reviews_test.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_message"
).show(10, truncate=False)

Review count: 99224
Invalid review IDs: 0
Invalid order IDs: 0
+--------------------------------+--------------------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_id                       |order_id                        |review_score|review_comment_message                                                                                                                                                        |
+--------------------------------+--------------------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |NULL                                                                                     

In [19]:
silver_reviews = transform_silver(
    corrected_bronze_reviews,
    "olist_order_reviews_dataset"
)

silver_reviews.write.mode("overwrite").parquet(
    str(SILVER / "olist_order_reviews_dataset")
)

silver_dfs["olist_order_reviews_dataset"] = (
    spark.read.parquet(
        str(SILVER / "olist_order_reviews_dataset")
    )
)

print(
    "Corrected Silver reviews:",
    silver_dfs["olist_order_reviews_dataset"].count()
)

NameError: name 'corrected_bronze_reviews' is not defined